# Lab 5 — 텍스트의 엔트로피와 Huffman 코드

**확률통계 · Topic 5 · 부산대학교 정보컴퓨터공학부**

---

### 오늘의 목표

1. 텍스트에서 **글자 빈도를 세어 엔트로피**를 직접 계산한다.
2. **Huffman 코드**를 만들어 평균 부호 길이와 $H$ 를 비교한다 (Shannon의 한계 확인).
3. **KL divergence**를 계산하고 **비대칭**을 눈으로 확인한다.

⏱ **예상 소요 시간: 35분**

> 📕 이 Topic는 주교재에 없는 내용이다. 슬라이드 마지막의 참고자료를 함께 보면 좋다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import heapq

rng = np.random.default_rng(20260302)
print("준비 완료")

## Part 1. 엔트로피 계산기 만들기

$$H(X) = -\sum_x p(x) \log_2 p(x)$$

⚠️ $p(x) = 0$ 인 항은 그냥 건너뛴다. ($0 \log 0 = 0$ 으로 정의한다)

### 실습 1

In [ ]:
def entropy(probs):
    p = np.asarray(probs, dtype=float)
    p = p[p > 0]                  # 0인 항은 제외
    # TODO 1: 엔트로피를 계산해 돌려주세요.  힌트: -(p * np.log2(p)).sum()
    return 0.0


print("공정한 동전     :", entropy([0.5, 0.5]))
print("치우친 동전 0.9 :", round(entropy([0.9, 0.1]), 4))
print("공정한 주사위   :", round(entropy([1 / 6] * 6), 4))
print("항상 같은 값    :", entropy([1.0, 0.0]))

> 공정한 동전은 1 bit, 치우친 동전은 0.469 bits, 결과가 하나뿐이면 0 bits.
> **불확실할수록 크다**는 감각을 확인하자.

## Part 2. 실제 텍스트의 엔트로피

영어 텍스트에서 글자 빈도를 세어 엔트로피를 구한다.

### 실습 2 — 빈도에서 확률로

In [ ]:
text = ("""the probability of an event is a number between zero and one
the larger the probability the more likely the event is to occur
statistics is the science of learning from data
information theory measures how surprising an outcome is"""
        .lower().replace("\n", " "))

text = "".join(ch for ch in text if ch.isalpha() or ch == " ")

counts = Counter(text)
total = sum(counts.values())
symbols = sorted(counts)

# TODO 2: 각 글자의 확률(= 등장 횟수 / 전체 길이)을 배열로 만드세요
probs = np.zeros(len(symbols))

print("글자 종류:", len(symbols), "/ 전체 길이:", total)
print("가장 흔한 5개:", counts.most_common(5))
print(f"엔트로피 H = {entropy(probs):.4f} bits/char")
print(f"균등분포라면 = {np.log2(len(symbols)):.4f} bits/char")

🤔 실제 텍스트의 엔트로피가 균등분포보다 **작다**.
글자가 고르게 나오지 않기 때문이다(`e`, `t`, 공백이 많다).

**이 차이가 곧 압축 가능한 여지**다.

## Part 3. Huffman 코드 — 이론 한계에 얼마나 가까운가

자주 나오는 글자에 짧은 부호를 준다. 아래 코드는 그대로 실행만 하면 된다.

In [ ]:
def huffman(symbols, probs):
    # (확률, 순번, {글자: 부호}) 를 우선순위 큐에 넣고 작은 것 둘씩 합친다
    heap = [(p, i, {s: ""}) for i, (s, p) in enumerate(zip(symbols, probs))]
    heapq.heapify(heap)
    counter = len(symbols)
    while len(heap) > 1:
        p1, _, c1 = heapq.heappop(heap)
        p2, _, c2 = heapq.heappop(heap)
        merged = {s: "0" + code for s, code in c1.items()}
        merged.update({s: "1" + code for s, code in c2.items()})
        heapq.heappush(heap, (p1 + p2, counter, merged))
        counter += 1
    return heap[0][2]


codebook = huffman(symbols, probs)
for s in sorted(symbols, key=lambda x: -counts[x])[:6]:
    label = "(공백)" if s == " " else s
    print(f"{label:>6} p={counts[s] / total:.4f}  code={codebook[s]}  길이 {len(codebook[s])}")

### 실습 3 — 평균 부호 길이와 엔트로피 비교

In [ ]:
H = entropy(probs)

# TODO 3: 평균 부호 길이 = sum(확률 x 부호 길이) 를 계산하세요
#         힌트: sum(probs[i] * len(codebook[s]) for i, s in enumerate(symbols))
avg_len = 0.0

print(f"엔트로피      H = {H:.4f} bits/char   <- 이론적 하한")
print(f"Huffman 평균 길이 = {avg_len:.4f} bits/char")
print(f"고정길이 부호라면 = {np.ceil(np.log2(len(symbols))):.0f} bits/char")

> **Huffman은 엔트로피보다 짧아질 수 없다.** 하지만 아주 가깝다.
> Shannon의 정리가 실제로 지켜지는 것을 확인했다.

## Part 4. KL divergence — 두 분포는 얼마나 다른가

$$D_{KL}(p \,\|\, q) = \sum_x p(x) \log_2 \frac{p(x)}{q(x)}$$

### 실습 4 — KL 계산기와 비대칭 확인

In [ ]:
def kl(p, q):
    p, q = np.asarray(p, float), np.asarray(q, float)
    mask = p > 0
    if np.any(q[mask] == 0):
        return np.inf
    # TODO 4: KL divergence 를 계산해 돌려주세요
    #         힌트: (p[mask] * np.log2(p[mask] / q[mask])).sum()
    return 0.0


p = np.array([0.7, 0.2, 0.1])
q = np.array([1 / 3, 1 / 3, 1 / 3])

print(f"D_KL(p||q) = {kl(p, q):.4f}")
print(f"D_KL(q||p) = {kl(q, p):.4f}   <- 같은 값인가요?")
print(f"D_KL(p||p) = {kl(p, p):.4f}")

⚠️ **`q`에 0이 있으면 KL이 무한대가 된다.**
Topic 2 스팸 필터에서 만난 그 문제다 — "한 번도 못 본 것에 확률 0을 주면 안 된다."

## Part 5. Cross-entropy는 언제 최소가 되나

예측 분포 $q$ 를 바꿔가며 $H(p, q)$ 를 그려보자.

### 실습 5

In [ ]:
def cross_entropy(p, q):
    p, q = np.asarray(p, float), np.asarray(q, float)
    mask = p > 0
    # TODO 5: cross-entropy 를 계산해 돌려주세요.  힌트: -(p[mask] * np.log2(q[mask])).sum()
    return 0.0


p = np.array([0.7, 0.2, 0.1])
alphas = np.linspace(0.02, 1.0, 100)
ce = [cross_entropy(p, a * p + (1 - a) * np.array([1 / 3, 1 / 3, 1 / 3])) for a in alphas]

plt.figure(figsize=(7, 4))
plt.plot(alphas, ce, lw=2, label="H(p, q)")
plt.axhline(entropy(p), color="red", ls="--", label=f"H(p) = {entropy(p):.3f}")
plt.xlabel("q gets closer to p  ->")
plt.ylabel("bits")
plt.title("Cross-entropy is minimized when q = p")
plt.legend()
plt.show()

🎯 **cross-entropy는 $q = p$ 일 때 최소가 되고, 그 최솟값이 $H(p)$ 다.**

이것이 딥러닝이 cross-entropy loss를 쓰는 이유의 전부다.
손실을 줄이라고 시키면 모델은 **예측 분포를 정답 분포에 맞추게** 된다.

$$\mathcal{L} = H(p, q) = \underbrace{H(p)}_{\text{못 줄이는 부분}} + \underbrace{D_{KL}(p\|q)}_{\text{줄일 수 있는 부분}}$$

---

## 마무리 — 자가 점검

- [ ] 확률분포에서 엔트로피를 계산할 수 있다
- [ ] Huffman 평균 길이가 $H$ 보다 짧아질 수 없음을 확인했다
- [ ] KL이 **비대칭**임을 직접 확인했다
- [ ] $q$ 에 0이 있으면 KL이 무한대가 되는 문제를 보았다
- [ ] cross-entropy가 $q = p$ 에서 최소가 됨을 확인했다

**오늘 배운 것을 한 문장으로.**

> (여기에 작성)

### 📌 이번 주 과제 대신 — **미니 프로젝트 1** 팀 구성과 주제 선정을 마칠 것